In [0]:
from pyspark.sql import functions as F

customer_features = spark.table("workspace.kkbox.customer_features_ml")

expiry_audit = (
    customer_features
    .groupBy("is_churn")
    .agg(
        F.count("*").alias("customer_count"),
        F.min("latest_expiry_date").alias("min_expiry_date"),
        F.max("latest_expiry_date").alias("max_expiry_date"),
        F.avg(
            F.when(
                F.col("latest_expiry_date").isNotNull(),
                F.datediff(
                    F.lit("2017-03-31"),
                    F.col("latest_expiry_date")
                )
            )
        ).alias("avg_days_from_observation_end")
    )
    .orderBy("is_churn")
)

display(expiry_audit)

In [0]:
transaction_timing_audit = (
    customer_features
    .agg(
        F.count("*").alias("total_customers"),
        F.sum(
            F.when(
                F.col("latest_transaction_date") > F.lit("2017-02-28"),
                1
            ).otherwise(0)
        ).alias("transactions_after_2017_02_28"),
        F.sum(
            F.when(
                F.col("latest_transaction_date") > F.lit("2017-03-31"),
                1
            ).otherwise(0)
        ).alias("transactions_after_2017_03_31"),
        F.min("latest_transaction_date").alias("min_transaction_date"),
        F.max("latest_transaction_date").alias("max_transaction_date")
    )
)

display(transaction_timing_audit)

In [0]:
march_transaction_audit = (
    customer_features
    .agg(
        F.count("*").alias("total_customers"),
        F.sum(
            F.when(
                F.col("latest_transaction_date") >= F.lit("2017-03-01"),
                1
            ).otherwise(0)
        ).alias("customers_with_march_transactions"),
        F.sum(
            F.when(
                F.col("latest_expiry_date") >= F.lit("2017-03-01"),
                1
            ).otherwise(0)
        ).alias("customers_with_march_expiry")
    )
)

display(march_transaction_audit)

In [0]:
from pyspark.sql import functions as F

original_transactions_path = "/Volumes/workspace/kkbox/raw_data/transactions.csv"

transactions_original_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(original_transactions_path)
)

transaction_audit = (
    transactions_original_df
    .agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("msno").alias("unique_customers"),
        F.min(
            F.to_date("transaction_date", "yyyyMMdd")
        ).alias("earliest_transaction_date"),
        F.max(
            F.to_date("transaction_date", "yyyyMMdd")
        ).alias("latest_transaction_date"),
        F.min(
            F.to_date("membership_expire_date", "yyyyMMdd")
        ).alias("earliest_expiry_date"),
        F.max(
            F.to_date("membership_expire_date", "yyyyMMdd")
        ).alias("latest_expiry_date")
    )
)

display(transaction_audit)

In [0]:
from pyspark.sql import functions as F

customer_scope = spark.table("workspace.kkbox.customer_scope")

original_transactions = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/workspace/kkbox/raw_data/transactions.csv")
)

transactions_clean_v1 = (
    original_transactions
    .join(
        customer_scope.select("msno"),
        on="msno",
        how="inner"
    )
    .select(
        "msno",
        "payment_method_id",
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "transaction_date",
        "membership_expire_date",
        "is_cancel",

        F.to_date(
            F.col("transaction_date").cast("string"),
            "yyyyMMdd"
        ).alias("transaction_date_clean"),

        F.to_date(
            F.col("membership_expire_date").cast("string"),
            "yyyyMMdd"
        ).alias("membership_expire_date_clean")
    )
)

display(transactions_clean_v1.limit(10))

In [0]:
transaction_v1_validation = (
    transactions_clean_v1
    .agg(
        F.count("*").alias("total_transaction_rows"),
        F.countDistinct("msno").alias("unique_customers"),
        F.min("transaction_date_clean").alias("earliest_transaction_date"),
        F.max("transaction_date_clean").alias("latest_transaction_date"),
        F.sum(
            F.when(
                F.col("transaction_date_clean").isNull(),
                1
            ).otherwise(0)
        ).alias("null_transaction_dates")
    )
)

display(transaction_v1_validation)

In [0]:
transactions_clean_v1.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.transactions_clean_v1")

In [0]:
spark.table("workspace.kkbox.transactions_clean_v1").count()

In [0]:
from pyspark.sql import functions as F

transactions_clean_v1 = spark.table(
    "workspace.kkbox.transactions_clean_v1"
)

transaction_features_v1 = (
    transactions_clean_v1
    .groupBy("msno")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("actual_amount_paid").alias("total_amount_paid"),
        F.avg("actual_amount_paid").alias("avg_amount_paid"),
        F.sum("plan_list_price").alias("total_plan_price"),
        F.avg("payment_plan_days").alias("avg_plan_days"),
        F.sum(
            F.when(F.col("is_auto_renew") == 1, 1).otherwise(0)
        ).alias("auto_renew_count"),
        F.sum(
            F.when(F.col("is_cancel") == 1, 1).otherwise(0)
        ).alias("cancel_count"),
        F.max("transaction_date_clean").alias("latest_transaction_date"),
        F.max("membership_expire_date_clean").alias("latest_expiry_date")
    )
)

display(transaction_features_v1.limit(10))

In [0]:
transaction_features_v1_validation = (
    transaction_features_v1
    .agg(
        F.count("*").alias("customers_with_transactions"),
        F.countDistinct("msno").alias("unique_customers"),
        F.min("transaction_count").alias("min_transaction_count"),
        F.max("transaction_count").alias("max_transaction_count"),
        F.min("total_amount_paid").alias("min_total_amount_paid"),
        F.max("total_amount_paid").alias("max_total_amount_paid"),
        F.max("latest_transaction_date").alias("latest_transaction_date")
    )
)

display(transaction_features_v1_validation)

In [0]:
transaction_features_v1.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.transaction_features_v1")

In [0]:
spark.table("workspace.kkbox.transaction_features_v1").count()

In [0]:
from pyspark.sql import functions as F

transaction_timing_v1 = (
    spark.table("workspace.kkbox.transactions_clean_v1")
    .agg(
        F.count("*").alias("total_transactions"),

        F.sum(
            F.when(
                F.col("transaction_date_clean")
                > F.col("membership_expire_date_clean"),
                1
            ).otherwise(0)
        ).alias("transactions_after_expiry"),

        F.sum(
            F.when(
                F.col("transaction_date_clean")
                == F.col("membership_expire_date_clean"),
                1
            ).otherwise(0)
        ).alias("transactions_on_expiry"),

        F.sum(
            F.when(
                F.col("membership_expire_date_clean").isNull(),
                1
            ).otherwise(0)
        ).alias("null_expiry_dates")
    )
)

display(transaction_timing_v1)

In [0]:
from pyspark.sql import functions as F

february_expiry_audit = (
    spark.table("workspace.kkbox.transactions_clean_v1")
    .filter(
        (F.col("membership_expire_date_clean") >= F.lit("2017-02-01")) &
        (F.col("membership_expire_date_clean") <= F.lit("2017-02-28"))
    )
    .groupBy("msno")
    .agg(
        F.count("*").alias("transactions_with_feb_expiry"),
        F.min("membership_expire_date_clean").alias("first_feb_expiry"),
        F.max("membership_expire_date_clean").alias("last_feb_expiry")
    )
)

display(
    february_expiry_audit
    .agg(
        F.count("*").alias("customers_with_february_expiry"),
        F.countDistinct("msno").alias("unique_customers"),
        F.min("first_feb_expiry").alias("earliest_feb_expiry"),
        F.max("last_feb_expiry").alias("latest_feb_expiry")
    )
)

In [0]:
from pyspark.sql import functions as F

customer_expiry = (
    spark.table("workspace.kkbox.transactions_clean_v1")
    .filter(
        (F.col("membership_expire_date_clean") >= F.lit("2017-02-01")) &
        (F.col("membership_expire_date_clean") <= F.lit("2017-02-28"))
    )
    .groupBy("msno")
    .agg(
        F.max("membership_expire_date_clean").alias("label_expiry_date")
    )
)

transaction_cutoff_audit = (
    spark.table("workspace.kkbox.transactions_clean_v1")
    .join(
        customer_expiry,
        on="msno",
        how="inner"
    )
    .agg(
        F.count("*").alias("total_transactions"),

        F.sum(
            F.when(
                F.col("transaction_date_clean") > F.col("label_expiry_date"),
                1
            ).otherwise(0)
        ).alias("transactions_after_label_expiry"),

        F.sum(
            F.when(
                F.col("transaction_date_clean") <= F.col("label_expiry_date"),
                1
            ).otherwise(0)
        ).alias("transactions_on_or_before_label_expiry"),

        F.max("transaction_date_clean").alias("latest_transaction_date")
    )
)

display(transaction_cutoff_audit)

In [0]:
from pyspark.sql import functions as F

transactions_v1 = spark.table(
    "workspace.kkbox.transactions_clean_v1"
)

customer_expiry = (
    transactions_v1
    .filter(
        (F.col("membership_expire_date_clean") >= F.lit("2017-02-01")) &
        (F.col("membership_expire_date_clean") <= F.lit("2017-02-28"))
    )
    .groupBy("msno")
    .agg(
        F.max("membership_expire_date_clean").alias("label_expiry_date")
    )
)

transactions_ml_safe = (
    transactions_v1
    .join(
        customer_expiry,
        on="msno",
        how="inner"
    )
    .filter(
        F.col("transaction_date_clean") <= F.col("label_expiry_date")
    )
)

display(transactions_ml_safe.limit(10))

In [0]:
transaction_ml_safe_validation = (
    transactions_ml_safe
    .agg(
        F.count("*").alias("total_transaction_rows"),
        F.countDistinct("msno").alias("unique_customers"),
        F.max("transaction_date_clean").alias("latest_transaction_date"),
        F.sum(
            F.when(
                F.col("transaction_date_clean") > F.col("label_expiry_date"),
                1
            ).otherwise(0)
        ).alias("post_expiry_transactions")
    )
)

display(transaction_ml_safe_validation)

In [0]:
transaction_features_ml = (
    transactions_ml_safe
    .groupBy("msno")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("actual_amount_paid").alias("total_amount_paid"),
        F.avg("actual_amount_paid").alias("avg_amount_paid"),
        F.sum("plan_list_price").alias("total_plan_price"),
        F.avg("payment_plan_days").alias("avg_plan_days"),
        F.sum(
            F.when(F.col("is_auto_renew") == 1, 1).otherwise(0)
        ).alias("auto_renew_count"),
        F.sum(
            F.when(F.col("is_cancel") == 1, 1).otherwise(0)
        ).alias("cancel_count"),
        F.max("transaction_date_clean").alias("latest_transaction_date"),
        F.max("membership_expire_date_clean").alias("latest_expiry_date")
    )
)

display(transaction_features_ml.limit(10))

In [0]:
transaction_features_ml_validation = (
    transaction_features_ml
    .agg(
        F.count("*").alias("customers_with_transactions"),
        F.countDistinct("msno").alias("unique_customers"),
        F.min("transaction_count").alias("min_transaction_count"),
        F.max("transaction_count").alias("max_transaction_count"),
        F.min("total_amount_paid").alias("min_total_amount_paid"),
        F.max("total_amount_paid").alias("max_total_amount_paid")
    )
)

display(transaction_features_ml_validation)

In [0]:
transaction_features_ml.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.transaction_features_ml")

In [0]:
spark.table("workspace.kkbox.transaction_features_ml").count()